# MovieLens Databricks Asset Bundle

A Databricks Asset Bundle is a source-controlled project containing application code, Databricks resource definitions, deployment settings, and environment-specific values. It makes validation, deployment, execution, and updates repeatable instead of requiring a manually configured workspace job.

This bundle deploys one four-task job: ingest movies, ingest ratings, calculate popular movies, then display and optionally publish the result through JDBC.

## Contents and workflow

```text
movielens/
├── databricks.yml                 # variables, workspace and dev/prod targets
├── resources/movielens_job.yml   # job, tasks, dependencies and parameters
└── src/
    ├── 01_ingest_movies.py
    ├── 02_ingest_ratings.py
    ├── 03_popular_movies.py
    └── 04_publish_jdbc.py

ingest_movies ──┐
                 ├──> popular_movies ──> publish_popular_movies
ingest_ratings ─┘
```

These `.py` files are Databricks source notebooks. The ingestion notebooks expose `input_path` and `output_path` with `dbutils.widgets.text`. The third task joins on `movie_id` and keeps movies rated by at least **100 distinct users** with an average rating of **4.0 or higher**.

## 1. Install the CLI and authenticate

Run command blocks in PowerShell. The current CLI is a standalone executable, not the legacy Python `databricks-cli` package.

```powershell
winget install --id Databricks.DatabricksCLI --exact
databricks version
databricks configure --host https://dbc-d2625d47-674e.cloud.databricks.com --profile course-free
databricks current-user me --profile course-free
```

Restart PowerShell after installation if the command is not found. `configure` prompts for the PAT and stores the profile in `%USERPROFILE%\.databrickscfg`; do not put the PAT in this notebook or bundle.

## 2. Verify the actual Unity Catalog paths

The commands below were used to inspect this workspace rather than assuming `main`.

```powershell
databricks catalogs list --profile course-free
databricks schemas list workspace --profile course-free
databricks volumes list workspace default --profile course-free
databricks fs ls dbfs:/Volumes/workspace/default/movielens/bronze --profile course-free
databricks fs ls dbfs:/Volumes/workspace/default/movielens/bronze/movies --profile course-free
databricks fs ls dbfs:/Volumes/workspace/default/movielens/bronze/ratings --profile course-free
```

Verified inputs:

```text
/Volumes/workspace/default/movielens/bronze/movies/movies.csv
/Volumes/workspace/default/movielens/bronze/ratings/ratings.csv
```

## 3. Understand parameters and environments

`databricks.yml` declares deployment-time variables. `movielens_job.yml` passes them to notebook widgets through `base_parameters`. Development and production use the same code but different Parquet roots:

```text
dev  -> /Volumes/workspace/default/movielens/silver/dev
prod -> /Volumes/workspace/default/movielens/silver/prod
```

The fourth task always reads and displays popular movies. JDBC writing defaults to `false`, so the tested workflow succeeds without an external database.

## 4. Validate the development bundle

There is no separate compile command. `bundle validate` resolves configuration and checks it against the authenticated workspace.

```powershell
Set-Location C:\Course\DataBricks\movielens
databricks bundle validate --target dev --profile course-free
databricks bundle summary --target dev --profile course-free
```

The verified remote root is `/Workspace/Users/ggknco@gmail.com/.bundle/movielens/dev`.

## 5. Optional deployment-time overrides

The committed defaults already match this workspace. For another directory, set overrides before both validation and deployment in the same PowerShell session.

```powershell
$env:BUNDLE_VAR_source_root = "/Volumes/workspace/default/movielens/bronze"
$env:BUNDLE_VAR_silver_root = "/Volumes/workspace/default/movielens/silver/dev"
databricks bundle validate --target dev --profile course-free
databricks bundle deploy --target dev --profile course-free

Remove-Item Env:BUNDLE_VAR_source_root -ErrorAction SilentlyContinue
Remove-Item Env:BUNDLE_VAR_silver_root -ErrorAction SilentlyContinue
```

## 6. Deploy and run development

Deployment uploads source files and creates or updates the bundle-managed job. Running by resource key executes all four tasks in dependency order.

```powershell
databricks bundle deploy --target dev --profile course-free
databricks bundle run movielens_pipeline --target dev --profile course-free
```

Expected final state: `TERMINATED SUCCESS`. The tested deployment created `movielens-dev`, and all four tasks completed successfully.

## 7. Verify the outputs

```powershell
databricks fs ls dbfs:/Volumes/workspace/default/movielens/silver/dev --profile course-free
databricks fs ls dbfs:/Volumes/workspace/default/movielens/silver/dev/movies --profile course-free
databricks fs ls dbfs:/Volumes/workspace/default/movielens/silver/dev/ratings --profile course-free
databricks fs ls dbfs:/Volumes/workspace/default/movielens/silver/dev/popular_movies --profile course-free
```

Each output directory should contain `_SUCCESS` and Parquet data files. The successful development run created all three directories.

## 8. Validate, deploy, and run production

Production is a distinct deployment. Review its resolved summary before deployment.

```powershell
Remove-Item Env:BUNDLE_VAR_silver_root -ErrorAction SilentlyContinue
databricks bundle validate --target prod --profile course-free
databricks bundle summary --target prod --profile course-free
databricks bundle deploy --target prod --profile course-free
databricks bundle run movielens_pipeline --target prod --profile course-free
databricks fs ls dbfs:/Volumes/workspace/default/movielens/silver/prod --profile course-free
```

These production commands are course instructions; the completed verification run used the development target.

## 9. Enable optional JDBC publishing

Create secrets interactively. The JDBC username and password are retrieved by `dbutils.secrets.get`, never stored in YAML or notebooks.

```powershell
databricks secrets create-scope movielens --profile course-free
databricks secrets put-secret movielens jdbc-user --profile course-free
databricks secrets put-secret movielens jdbc-password --profile course-free

$env:BUNDLE_VAR_jdbc_enabled = "true"
$env:BUNDLE_VAR_jdbc_url = "jdbc:postgresql://SERVER:5432/DATABASE"
$env:BUNDLE_VAR_jdbc_table = "popular_movies"
$env:BUNDLE_VAR_jdbc_secret_scope = "movielens"
$env:BUNDLE_VAR_jdbc_user_key = "jdbc-user"
$env:BUNDLE_VAR_jdbc_password_key = "jdbc-password"
databricks bundle validate --target dev --profile course-free
databricks bundle deploy --target dev --profile course-free
databricks bundle run movielens_pipeline --target dev --profile course-free
```

The corresponding JDBC driver must be available in the Databricks Runtime. With `jdbc_enabled=false`, the notebook only displays data and prints a skip message.

## 10. Bundle lifecycle commands

```powershell
# Inspect deployed resources
databricks bundle summary --target dev --profile course-free

# Validate and redeploy after changing code or YAML
databricks bundle validate --target dev --profile course-free
databricks bundle deploy --target dev --profile course-free

# Remove bundle-managed development workspace resources
databricks bundle destroy --target dev --profile course-free
```

`bundle destroy` asks for confirmation and removes bundle-managed workspace resources. It does not remove Parquet data written to the volume.